# GSEA

Volcano and heatmap plots are used to display differentially expressed genes and their expression patterns.


In [ ]:
import anndata
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from adjustText import adjust_text
import adjustText
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import gseapy as gp
import omicverse as ov
import re
from pathlib import Path
import os
from datetime import date
from scipy.sparse import issparse
ov.plot_set()
%load_ext autoreload
%autoreload 2

In [ ]:
mpl.rcParams["font.family"] = "Arial"          
mpl.rcParams["font.sans-serif"] = ["Arial"]    
mpl.rcParams["pdf.fonttype"] = 42              
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["savefig.dpi"] = 300

mpl.rcParams["figure.dpi"]  = 300

In [ ]:
# 创建自定义 colormap
colors = ["#22456F", "#528FAC","#AECAD8","#F2F2F2","#F7D29D","#EEB84B", "#E87A2D"]  # 示例颜色
cmap_custom = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)

In [ ]:
# Base directory
file_path = r"F:\Data\snRNAseq\PlaceCell\Results\PathwayEnrichment\GSEA"
today = date.today().strftime("%Y%m%d")
# Input DEG file

deg_file = "place_silent_deseq2_20260128.csv"
# Load data
rnk = pd.read_csv(
    os.path.join(file_path,deg_file),
    index_col=0
)[['log2FoldChange']]


In [ ]:
output_path = r"F:\Data\snRNAseq\PlaceCell\Results\PathwayEnrichment\GSEA\log2FC"

os.makedirs(output_path, exist_ok=True)

In [ ]:
rnk.head()

In [ ]:
# rnk: index=gene, column=stat  →  gene / score
rnk2 = rnk.reset_index()
rnk2.columns = ["gene", "score"]

# 排序（非常重要）
rnk2 = rnk2.sort_values("score", ascending=False)

# 基本 sanity check
assert rnk2.shape[1] == 2
assert rnk2["gene"].is_unique
print(rnk2.head())
print(rnk2["score"].min(), rnk2["score"].max())


In [ ]:
rnk2.shape

In [ ]:
pathway = "GO_Biological_Process_2025"
# pathway = "KEGG_2019_Mouse"
# pathway = "Reactome_Pathways_2024"
pathway_dict=ov.utils.geneset_prepare(fr'F:\Code\Python\snRNAseq\genesets\{pathway}.txt',organism='Mouse')


In [ ]:
pre_res = gp.prerank(rnk=rnk2, # or rnk = rnk,
                     gene_sets= pathway_dict,
                     threads=4,
                     min_size=5,
                     max_size=1000,
                     permutation_num=1000, # reduce number to speed up testing
                     outdir=None, # don't write to disk
                     seed=6,
                     verbose=True, # see what's going on behind the scenes
                    )

In [ ]:
pre_res.res2d.head(10)

In [ ]:
terms = pre_res.res2d.Term
axs = pre_res.plot(terms=terms[1]) # v1.0.5
# plt.savefig(f"{output_path}/{pathway}_Term.png",
#             dpi=300, bbox_inches="tight")

In [ ]:
axs = pre_res.plot(terms=terms[1:5],
                   #legend_kws={'loc': (1.2, 0)}, # set the legend loc
                   show_ranking=True, # whether to show the second yaxis
                   figsize=(3,4)
                  )
# plt.savefig(f"{output_path}/{pathway}_MultiTerm.png",
#             dpi=300, bbox_inches="tight")

In [ ]:
from gseapy import dotplot
# to save your figure, make sure that ``ofname`` is not None
ax = dotplot(pre_res.res2d,
             column="FDR q-val",
             title= pathway,
             cmap=plt.cm.viridis,
             top_term=10,
             size=6, # adjust dot size
             figsize=(4,5), cutoff=0.25, show_ring=False)
# plt.savefig(f"{output_path}/{pathway}_Bubble.png",
#             dpi=300, bbox_inches="tight")

In [ ]:
pre_res.res2d.to_csv(
    os.path.join(output_path, "place_silent_GSEA_GO_results.csv"),
    index=False
)
# Reactome_Pathways_2024
